<a href="https://www.kaggle.com/code/shivamvachhnai/sentiment-analysis-on-imdb-dataset?scriptVersionId=315236406" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
import pandas as pd 
import numpy as np
import re

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/waseemalastal/imdb-dataset/IMDB Dataset.csv')
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)
df.duplicated().sum()

# Text Preprocessing

In [ ]:
# Lowercasing reviews  
df['review'] = df['review'].apply(lambda x:x.lower()) 

In [ ]:
# Remove HTML tags
def remove_html_tags(text):
    pattern=re.compile('<.*?>')
    return pattern.sub(r'',text)

df['review'] = df['review'].apply(remove_html_tags)
df['review'][0]

In [ ]:
# Remove puctuation
import string 
exclude = string.punctuation
def remove_punc(text):
    return text.translate(str.maketrans('','',exclude))

df['review'] = df['review'].apply(remove_punc)
df['review'][0]

In [ ]:
# Remove stopwords 
from nltk.corpus import stopwords
stop_words = stopwords.words('english')

def remove_sw(text):
    filter_words = [w for w in text.split() if w not in stop_words]
    return ' '.join(filter_words)

df['review'] = df['review'].apply(remove_sw)
df['review'][0]

In [ ]:
# Stemming 
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()
def stem_words(text):
    return ' '.join([ps.stem(word) for word in text.split()])

df['review'] = df['review'].apply(stem_words)
df['review'][0]

# Train Test Split

In [ ]:
temp_df=df[:10000]
X=temp_df.iloc[:,0:1]
y=temp_df['sentiment'].apply(lambda x:0 if x == "negative" else 1)

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
X_train.shape

# Text Embedding & Modeling

In [ ]:
# BOW(Bag of words)
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer()

In [ ]:
X_train_bow = cv.fit_transform(X_train['review']).toarray()
X_test_bow= cv.transform(X_test['review']).toarray()

In [ ]:
X_train_bow.shape

In [ ]:
# Naive Bayes
from sklearn.naive_bayes import GaussianNB
gnb = GaussianNB()
gnb.fit(X_train_bow,y_train)

In [ ]:
# Model evaluation
y_pred = gnb.predict(X_test_bow)

from sklearn.metrics import classification_report,confusion_matrix,accuracy_score
print(classification_report(y_pred,y_test))
print(f"Accuracy = {accuracy_score(y_pred,y_test):.2f}")

In [ ]:
# Random Forest 
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()
rf.fit(X_train_bow,y_train)
y_pred = rf.predict(X_test_bow)
print(classification_report(y_pred,y_test))

In [ ]:
cv = CountVectorizer(ngram_range=(1,2),max_features=30000)

X_train_ngram = cv.fit_transform(X_train['review']).toarray()
X_test_ngram = cv.transform(X_test['review']).toarray()

rf = RandomForestClassifier()
rf.fit(X_train_ngram,y_train)
y_pred = rf.predict(X_test_ngram)
print(classification_report(y_pred,y_test))

# Using Tf-Idf

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(X_train['review']).toarray()
X_test_tfidf = tfidf.transform(X_test['review']).toarray()

rf = RandomForestClassifier()
rf.fit(X_train_tfidf,y_train)

y_pred = rf.predict(X_test_tfidf)
print(classification_report(y_pred,y_test))


# Using Word2Vec 

In [ ]:
import gensim
from gensim.utils import simple_preprocess
from nltk import sent_tokenize

story=[]

for doc in df['review']:
    raw_sent = sent_tokenize(doc)
    for sent in raw_sent:
        story.append(simple_preprocess(sent))


In [ ]:
model = gensim.models.Word2Vec(
    window=10,
    min_count=2,
    workers=4
)

In [ ]:
model.build_vocab(story)

In [ ]:
model.train(story,total_examples=model.corpus_count,epochs=model.epochs)

In [ ]:
def document_vector(doc):
    doc = [word for word in doc.split() if word in model.wv.index_to_key]
    return np.mean(model.wv[doc],axis=0)

In [ ]:
document_vector(df['review'][2])

In [ ]:
from tqdm import tqdm
X=[]
for doc in tqdm(df['review'].values):
    X.append(document_vector(doc))

In [ ]:
y = df['sentiment'].apply(lambda x:0 if x == "negative" else 1)

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
rf = RandomForestClassifier()
rf.fit(X_train,y_train)

y_pred = rf.predict(X_test)
print(classification_report(y_pred,y_test))